In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 18. Learning curves and the generalisation gap

Answers a question Dr Hunter raised twice and that the dissertation currently cannot answer:
**would more data help?**

He said it in two forms. On 22 October: *"maybe there's just too much variation in the data...
we haven't got enough data really that's accurate enough to really train it adequately."* And
when discussing the English comparison: *"if you train it on more data, then you expect to get
better results usually."*

A learning curve settles it. Train the same pipeline on 10%, 20% ... 100% of the training
split and plot held-out macro-F1 against training size. If the curve is still climbing at
100%, more data would help and that is a clean recommendation for future work. If it has
flattened, more data of the same kind would not, and the limit is the data's nature rather
than its quantity.

The same run also produces the **generalisation gap**, training accuracy minus test accuracy,
at every size. That is the number Hunter asked for directly: *"that's always worth noting
what the training error was."*

Hinglish and English are run side by side, so the curves are comparable.

**Runtime: about 3 minutes, CPU only.** The optional transformer section at the end needs a
GPU and about 25 minutes.

In [2]:
import os, sys, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

ROOT = Path(os.environ.get('HINGLISH_HATE_ROOT', r'H:\Data\msc-hinglish-hate'))
if not (ROOT / 'data').exists():
    ROOT = Path.cwd().parent
DATA, W = ROOT / 'data', ROOT / 'writing'
FIG = W / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
SEED = 42
FRACTIONS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.85, 1.0]
print('root:', ROOT)


def build_pipeline():
    """Same as notebook 17: no lexicon, so Hinglish and English are directly comparable."""
    feats = FeatureUnion([
        ('word', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2,
                                 sublinear_tf=True)),
    ])
    return Pipeline([('features', feats),
                     ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',
                                                random_state=SEED))])

root: /


In [3]:
def learning_curve(name, Xtr, ytr, Xte, yte, fractions=FRACTIONS, repeats=3):
    """Held-out score against training size, averaged over `repeats` subsamples."""
    Xtr = pd.Series(list(Xtr)).reset_index(drop=True)
    ytr = np.asarray(ytr)
    rows = []
    for f in fractions:
        n = int(round(f * len(ytr)))
        accs, f1s, tracc = [], [], []
        reps = 1 if f >= 0.999 else repeats
        for r in range(reps):
            if f >= 0.999:
                idx = np.arange(len(ytr))
            else:
                # stratified subsample, so the hate rate is held constant
                idx, _ = train_test_split(np.arange(len(ytr)), train_size=n,
                                          stratify=ytr, random_state=SEED + r)
            pipe = build_pipeline()
            pipe.fit(Xtr.iloc[idx], ytr[idx])
            pte = pipe.predict(Xte)
            f1s.append(f1_score(yte, pte, average='macro'))
            accs.append(accuracy_score(yte, pte))
            tracc.append(accuracy_score(ytr[idx], pipe.predict(Xtr.iloc[idx])))
        rows.append({'dataset': name, 'fraction': f, 'n_train': n,
                     'macro_f1': round(float(np.mean(f1s)), 4),
                     'macro_f1_sd': round(float(np.std(f1s)), 4),
                     'test_acc': round(float(np.mean(accs)), 4),
                     'train_acc': round(float(np.mean(tracc)), 4),
                     'gap': round(float(np.mean(tracc) - np.mean(accs)), 4)})
        print(f'  {name:22s} n={n:6d}  macro-F1 {np.mean(f1s):.3f}  gap {np.mean(tracc)-np.mean(accs):.3f}')
    return pd.DataFrame(rows)

In [4]:
curves = []

# Redirect DATA and W to the active Google Drive path
DRIVE_DIR = Path('/content/drive/MyDrive/dissertation')
DATA = DRIVE_DIR / 'data'
W = DRIVE_DIR

# ---- Hinglish, the frozen split -------------------------------------------
tr = pd.read_parquet(DATA / 'splits' / 'bohra_train.parquet')
te = pd.read_parquet(DATA / 'splits' / 'bohra_test.parquet')
print('Hinglish (Bohra)')
t0 = time.time()
curves.append(learning_curve('Hinglish (Bohra)', tr['text'], tr['label'].values,
                             te['text'], te['label'].values))

# ---- English, matched on size and balance, from notebook 17 ----------------
CSV = DATA / 'english' / 'davidson_labeled_data.csv'
if CSV.exists():
    eng = pd.read_csv(CSV)
    eng['label'] = (eng['class'] == 0).astype(int)
    eng = eng[['tweet', 'label']].rename(columns={'tweet': 'text'}).dropna()
    hate, notu = eng[eng.label == 1], eng[eng.label == 0]
    n_not = int(round(len(hate) * (1 - 0.363) / 0.363))
    m = (pd.concat([hate, notu.sample(n_not, random_state=SEED)])
           .sample(frac=1, random_state=SEED))
    mtr, mte = train_test_split(m, test_size=0.2, stratify=m['label'], random_state=SEED)
    print('\nEnglish (Davidson), matched')
    curves.append(learning_curve('English (Davidson)', mtr['text'], mtr['label'].values,
                                 mte['text'], mte['label'].values))
else:
    print('\nEnglish data not found. Run notebook 17 first to download it.')

lc = pd.concat(curves, ignore_index=True)
lc.to_csv(W / 'learning_curves.csv', index=False)
print(f'\n{time.time()-t0:.0f}s total')
display(lc)

Hinglish (Bohra)
  Hinglish (Bohra)       n=   366  macro-F1 0.562  gap 0.394
  Hinglish (Bohra)       n=   732  macro-F1 0.597  gap 0.360
  Hinglish (Bohra)       n=  1098  macro-F1 0.610  gap 0.339
  Hinglish (Bohra)       n=  1464  macro-F1 0.611  gap 0.332
  Hinglish (Bohra)       n=  1830  macro-F1 0.619  gap 0.320
  Hinglish (Bohra)       n=  2195  macro-F1 0.615  gap 0.321
  Hinglish (Bohra)       n=  2561  macro-F1 0.622  gap 0.309
  Hinglish (Bohra)       n=  3110  macro-F1 0.616  gap 0.311
  Hinglish (Bohra)       n=  3659  macro-F1 0.621  gap 0.301

English data not found. Run notebook 17 first to download it.

76s total


,dataset,fraction,n_train,macro_f1,macro_f1_sd,test_acc,train_acc,gap
0,Hinglish (Bohra),0.10,366,0.5620,0.0050,0.5923,0.9863,0.3940
1,Hinglish (Bohra),0.20,732,0.5975,0.0088,0.6222,0.9822,0.3600
2,Hinglish (Bohra),0.30,1098,0.6104,0.0043,0.6368,0.9760,0.3392
3,Hinglish (Bohra),0.40,1464,0.6107,0.0067,0.6346,0.9663,0.3317
4,Hinglish (Bohra),0.50,1830,0.6193,0.0083,0.6412,0.9614,0.3202
5,Hinglish (Bohra),0.60,2195,0.6150,0.0104,0.6368,0.9576,0.3208
6,Hinglish (Bohra),0.70,2561,0.6216,0.0058,0.6426,0.9518,0.3092
7,Hinglish (Bohra),0.85,3110,0.6165,0.0038,0.6372,0.9479,0.3108
8,Hinglish (Bohra),1.00,3659,0.6212,0.0000,0.6437,0.9442,0.3005


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
colours = {'Hinglish (Bohra)': '#d1495b', 'English (Davidson)': '#2b7bba'}

for name, g in lc.groupby('dataset'):
    c = colours.get(name, '#666666')
    axes[0].plot(g['n_train'], g['macro_f1'], 'o-', color=c, label=name, markersize=4)
    axes[0].fill_between(g['n_train'], g['macro_f1'] - g['macro_f1_sd'],
                         g['macro_f1'] + g['macro_f1_sd'], color=c, alpha=0.15)
    axes[1].plot(g['n_train'], g['train_acc'], 'o--', color=c, markersize=3,
                 label=f'{name}, training')
    axes[1].plot(g['n_train'], g['test_acc'], 'o-', color=c, markersize=4,
                 label=f'{name}, test')

axes[0].set_xlabel('training posts'); axes[0].set_ylabel('held-out macro-F1')
axes[0].set_title('Does more data help?', fontsize=10)
axes[0].legend(fontsize=8); axes[0].set_ylim(0.4, 0.9); axes[0].grid(alpha=0.25)

axes[1].set_xlabel('training posts'); axes[1].set_ylabel('accuracy')
axes[1].set_title('Training against held-out accuracy', fontsize=10)
axes[1].legend(fontsize=7); axes[1].set_ylim(0.5, 1.02); axes[1].grid(alpha=0.25)

fig.tight_layout()
fig.savefig(FIG / 'learning_curves.png', dpi=200, bbox_inches='tight'); plt.close(fig)
print('saved learning_curves.png')

saved learning_curves.png


In [6]:
# --- the slope at the end, which is what answers the question ---------------
print('Change in macro-F1 over the last 30% of training data:\n')
for name, g in lc.groupby('dataset'):
    g = g.sort_values('n_train')
    lo = g[g.fraction >= 0.7].iloc[0]
    hi = g.iloc[-1]
    delta = hi['macro_f1'] - lo['macro_f1']
    per1k = delta / max((hi['n_train'] - lo['n_train']) / 1000, 1e-9)
    print(f'  {name:22s} {lo["macro_f1"]:.3f} at n={int(lo["n_train"])}'
          f' -> {hi["macro_f1"]:.3f} at n={int(hi["n_train"])}'
          f'   ({delta:+.3f}, {per1k:+.3f} per 1,000 posts)')
print('\nA slope near zero means the curve has flattened and more data of the same kind'
      '\nwould not help. A clearly positive slope means it would.')

Change in macro-F1 over the last 30% of training data:

  Hinglish (Bohra)       0.622 at n=2561 -> 0.621 at n=3659   (-0.000, -0.000 per 1,000 posts)

A slope near zero means the curve has flattened and more data of the same kind
would not help. A clearly positive slope means it would.


## Optional: the same curve for a transformer

Only worth running if the statistical curve leaves the question open. It needs a GPU and
about 25 minutes, because it trains XLM-RoBERTa once per training size.

Set `RUN_TRANSFORMER = True` to run it.

In [7]:
RUN_TRANSFORMER = False      # set True on the A5000

if RUN_TRANSFORMER:
    import torch, random, gc
    from torch.utils.data import Dataset, DataLoader
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              get_linear_schedule_with_warmup)
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    MODEL, EPOCHS, LR, BS = 'xlm-roberta-base', 3, 2e-5, 16

    class PostDataset(Dataset):
        def __init__(self, texts, labels, tok, max_len=128):
            self.t = list(texts); self.y = list(labels); self.tok = tok; self.m = max_len
        def __len__(self): return len(self.t)
        def __getitem__(self, i):
            e = self.tok(str(self.t[i]), truncation=True, padding='max_length',
                         max_length=self.m, return_tensors='pt')
            return {'input_ids': e['input_ids'].squeeze(0),
                    'attention_mask': e['attention_mask'].squeeze(0),
                    'labels': torch.tensor(self.y[i], dtype=torch.long)}

    def set_all_seeds(s):
        random.seed(s); np.random.seed(s); torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

    rows = []
    for f in [0.25, 0.5, 0.75, 1.0]:
        n = int(round(f * len(tr)))
        idx = (np.arange(len(tr)) if f >= 0.999 else
               train_test_split(np.arange(len(tr)), train_size=n,
                                stratify=tr['label'].values, random_state=SEED)[0])
        sub = tr.iloc[idx]
        set_all_seeds(SEED)
        tok = AutoTokenizer.from_pretrained(MODEL)
        dl_tr = DataLoader(PostDataset(sub['text'], sub['label'], tok),
                           batch_size=BS, shuffle=True)
        counts = np.bincount(sub['label'].values, minlength=2)
        wts = torch.tensor(len(sub) / (2.0 * np.maximum(counts, 1)),
                           dtype=torch.float).to(DEVICE)
        set_all_seeds(SEED)
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL, num_labels=2).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        total = len(dl_tr) * EPOCHS
        sch = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
        lossf = torch.nn.CrossEntropyLoss(weight=wts)
        for _ in range(EPOCHS):
            model.train()
            for b in dl_tr:
                lg = model(input_ids=b['input_ids'].to(DEVICE),
                           attention_mask=b['attention_mask'].to(DEVICE)).logits
                loss = lossf(lg, b['labels'].to(DEVICE)); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sch.step(); opt.zero_grad()

        def pred(frame):
            dl = DataLoader(PostDataset(frame['text'], frame['label'], tok), batch_size=BS*2)
            model.eval(); P, T = [], []
            with torch.no_grad():
                for b in dl:
                    lg = model(input_ids=b['input_ids'].to(DEVICE),
                               attention_mask=b['attention_mask'].to(DEVICE)).logits
                    P.extend(lg.argmax(1).cpu().numpy()); T.extend(b['labels'].numpy())
            return np.array(T), np.array(P)

        yte_t, pte_t = pred(te)
        ytr_t, ptr_t = pred(sub)
        rows.append({'dataset': 'Hinglish, XLM-R', 'fraction': f, 'n_train': n,
                     'macro_f1': round(f1_score(yte_t, pte_t, average='macro'), 4),
                     'test_acc': round(accuracy_score(yte_t, pte_t), 4),
                     'train_acc': round(accuracy_score(ytr_t, ptr_t), 4)})
        rows[-1]['gap'] = round(rows[-1]['train_acc'] - rows[-1]['test_acc'], 4)
        print(f"  n={n:5d}  macro-F1 {rows[-1]['macro_f1']:.3f}  gap {rows[-1]['gap']:.3f}")
        del model; gc.collect(); torch.cuda.empty_cache()

    tlc = pd.DataFrame(rows)
    tlc.to_csv(W / 'learning_curve_xlmr.csv', index=False)
    display(tlc)
else:
    print('transformer section skipped (RUN_TRANSFORMER = False)')

transformer section skipped (RUN_TRANSFORMER = False)


## How to use the result

Two sentences for the Discussion, filled in from the output above.

**If the Hinglish curve has flattened:** more Hinglish data of this kind would not raise the
score, so the limit lies in the labels or in the language rather than in the sample size.
That is a stronger conclusion than "it is a hard problem", and it supports the error analysis.

**If it is still climbing:** more data would help, which becomes a concrete recommendation
for future work and a fair explanation of the current result.

The second panel is the answer to his question about training accuracy. Where the training
curve sits high and flat while the test curve sits well below it and also flat, the model is
memorising, and the distance between the two lines is the share of what it learns that does
not transfer.